## Objectif : Selection du model
### sur un echantillon du jeu de donnée
#### pour ne pas faire les tests sur 290 millions de lignes
##### On entrainera ensuite le best_model sur le vrai dataset

## Suivi MLflow des entraînements partiels

Une campagne d'échantillonnage correspond à un **run parent**. Il archive la source journalière, le taux d'échantillonnage, les features et les bornes temporelles. Chaque combinaison modèle/hyperparamètres est un **run enfant**. Les modèles sont comparés sur la même validation temporelle et avec le même échantillon : la PR-AUC reste ainsi comparable.

### Connexion et initialisation MLflow
connecté au conteneur Postgres mlflow

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


sys.path.append(str(Path.cwd().parent))

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import psycopg
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid

from src.config import (
    HOST,
    NAME,
    USER,
    PASSWORD,
    PORT,
    URI,
    SPLIT_TRAIN_START,
    SPLIT_TRAIN_END,
    SPLIT_TEST_START,
    SPLIT_VAL_START,
    SPLIT_VAL_YEAR
)
from src.db import read_query

# Connexion MLflow la base PostgreSQL
mlflow_uri = URI
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("Prediction_Risque_Incendies")


print("Tracking URI MLflow configuré sur PostgreSQLavec l'expérience : Prediction_Risque_Incendies")



[2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Tracking URI MLflow configuré sur PostgreSQLavec l'expérience : Prediction_Risque_Incendies


### Chargement échantillonné depuis la base de données

Va chercher une vue ou un échantillon stratifié directement en SQL
en conservant tous les positifs $Y=1$ et un sous-échantillon contrôlé de négatifs $Y=0$ pour accélérer le Grid Search

In [2]:
# Contrat de la source journalière : une ligne commune-jour et des features
# calculées uniquement avec des données antérieures à date_jour.
# La table brute commune_jour ne suffit pas : elle ne contient que has_fire.
DAILY_FEATURE_SOURCE = 'incendies.commune_jour'
TARGET = 'target_occurrence'
DATE_COLUMN = 'date_jour'
ID_COLUMN = 'code_insee'
LEAKAGE_COLUMNS = {
    ID_COLUMN, DATE_COLUMN, TARGET, 'annee', 'has_fire', 'nb_incendies',
    'surface_brulee_ha', 'surface_parcourue', 'target_log_surface',
}

def get_conn():
    return psycopg.connect(host=HOST, port=PORT, user=USER, password=PASSWORD, dbname=NAME)

dataset_metadata = {
    'granularity': 'commune-jour',
    'source': DAILY_FEATURE_SOURCE,
    'sampling': 'all positives + deterministic hash sample of negatives',
}


# Échantillonnage journalier reproductible directement en base

In [4]:
# Échantillonnage journalier reproductible directement en base
NEGATIVE_SAMPLE_RATE = 0.005
HASH_MODULUS = 1_000_000
HASH_THRESHOLD = int(NEGATIVE_SAMPLE_RATE * HASH_MODULUS)
RANDOM_SEED = 42

def load_daily_sample(start_date, end_date):
    # Jointure avec commune pour injecter les descripteurs spatiaux
    # has_fire est filtré directement dans le WHERE
    query = f"""
        SELECT
            cj.date_jour,
            c.code_insee,
            c.population,
            c.superficie_hectare,
            c.densite,
            c.altitude_moyenne,
            c.latitude,
            c.longitude,
            cj.has_fire AS {TARGET}
        FROM {DAILY_FEATURE_SOURCE} cj
        JOIN incendies.commune c ON cj.id_commune = c.code_insee
        WHERE cj.{DATE_COLUMN} >= %s
          AND cj.{DATE_COLUMN} < %s
          AND (
              cj.has_fire = 1
              OR MOD((hashtextextended(cj.id_commune || cj.{DATE_COLUMN}::text, %s) & 9223372036854775807), %s) < %s
          )
    """
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(query, (start_date, end_date, RANDOM_SEED, HASH_MODULUS, HASH_THRESHOLD))
        columns = [column.name for column in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)

# Extractions séparées par période (aucun data leakage temporel)
df_train_raw = load_daily_sample('2006-01-01', '2023-01-01')
df_val_raw = load_daily_sample('2023-01-01', '2024-01-01')
df_test_raw = load_daily_sample('2024-01-01', '2026-01-01')

df_dataset = pd.concat([df_train_raw, df_val_raw, df_test_raw], ignore_index=True)
df_dataset[DATE_COLUMN] = pd.to_datetime(df_dataset[DATE_COLUMN])
df_dataset['annee'] = df_dataset[DATE_COLUMN].dt.year
df_dataset['mois'] = df_dataset[DATE_COLUMN].dt.month

# Vérification du contrat
missing = {ID_COLUMN, DATE_COLUMN, TARGET} - set(df_dataset.columns)
if missing:
    raise ValueError(f'La vue journalière ne respecte pas le contrat : {missing}')

X = [
    column for column in df_dataset.select_dtypes(include=np.number).columns
    if column not in LEAKAGE_COLUMNS and not column.startswith('target_')
]
y = TARGET

if not X:
    raise ValueError('Aucune feature numérique disponible.')

print(f'Échantillon extrait : {df_dataset.shape} | Features de base disponibles : {len(X)}')

UndefinedFunction: operator does not exist: integer = character varying
LINE 13:         JOIN incendies.commune c ON cj.id_commune = c.code_i...
                                                           ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.

In [ ]:
# =============================================================================
# CONSTANTES DE SPLIT TEMPOREL (Source unique de vérité)
# =============================================================================
# SPLIT_TRAIN_END  = pd.Timestamp('2022-12-31')
# SPLIT_VAL_START  = pd.Timestamp('2023-01-01')
# SPLIT_VAL_YEAR = 2023
# SPLIT_TEST_START = pd.Timestamp('2024-01-01')

mask_train = df_dataset[DATE_COLUMN] <= SPLIT_TRAIN_END
mask_val = (df_dataset[DATE_COLUMN] >= SPLIT_VAL_START) & (df_dataset[DATE_COLUMN] < SPLIT_TEST_START)
mask_test = df_dataset[DATE_COLUMN] >= SPLIT_TEST_START

X_train, y_train = df_dataset.loc[mask_train, X], df_dataset.loc[mask_train, y]
X_val,   y_val   = df_dataset.loc[mask_val, X],   df_dataset.loc[mask_val, y]
X_test,  y_test  = df_dataset.loc[mask_test, X],  df_dataset.loc[mask_test, y]

# Contrôle des volumes
def print_split_stats(name, y_sub):
    total = len(y_sub)
    positives = y_sub.sum()
    rate = (positives / total) * 100
    print(f"--- {name} : {total:,} lignes | Incendies : {positives:,} ({rate:.2f}%)".replace(",", " "))

assert df_dataset.loc[mask_train, DATE_COLUMN].max() < df_dataset.loc[mask_val, DATE_COLUMN].min()
assert df_dataset.loc[mask_val, DATE_COLUMN].max() < df_dataset.loc[mask_test, DATE_COLUMN].min()
print_split_stats(f"TRAIN (<= {SPLIT_TRAIN_END.date()})", y_train)
print_split_stats('VAL   (2023)', y_val)
print_split_stats(f"TEST  (>= {SPLIT_TEST_START})", y_test)

--- TRAIN (<= 2022) : 1 973 088 lignes | Incendies : 36 524 (1.85%)
--- VAL   (2023) : 116 064 lignes | Incendies : 2 282 (1.97%)
--- TEST  (>= 2024) : 232 128 lignes | Incendies : 3 289 (1.42%)


In [ ]:
# Échantillonnage pour le Grid Search

# Sous-échantillonnage stratifié par année sur le Train uniquement
train_grid_parts = []
df_train_only = df_dataset.loc[mask_train].copy()

for year, year_data in df_train_only.groupby('annee', observed=True):
    positives = year_data[year_data[y] == 1]
    negatives = year_data[year_data[y] == 0]
    negative_sample = negatives.sample(
        n=min(len(negatives), len(positives) * 10),
        random_state=42 + int(year),
    )
    train_grid_parts.append(pd.concat([positives, negative_sample]))

train_grid = (
    pd.concat(train_grid_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

X_search_train = train_grid[X]
y_search_train = train_grid[y]
search_pos_weight = (len(y_search_train) - y_search_train.sum()) / y_search_train.sum()

# La validation du Grid Search pointe directement sur le jeu de validation global
X_search_val = X_val
y_search_val = y_val

print(f"Train complet : {len(y_train):,} lignes")
print(f"Train Grid Search (1:10) : {len(train_grid):,} lignes (scale_pos_weight: {search_pos_weight:.2f})")
print(f"Validation Grid Search ({SPLIT_VAL_YEAR}) : {len(y_search_val):,} lignes")

Train complet : 1,973,088 lignes
Train Grid Search (1:10) : 401,764 lignes (scale_pos_weight: 10.00)
Validation Grid Search (2023) : 116,064 lignes


In [ ]:

# Exécution du Grid Search et sélection du meilleur modèle

model_grids = {
    'LightGBM': {
        'num_leaves': [31, 63],
        'max_depth': [-1, 8],
        'learning_rate': [0.05],
        'n_estimators': [300],
        'min_child_samples': [50],
    },
    'XGBoost': {
        'max_depth': [6, 8],
        'learning_rate': [0.05],
        'n_estimators': [300],
        'min_child_weight': [1, 5],
    },
}

parent_run = mlflow.start_run(run_name='benchmark_daily_sample')
mlflow.set_tags({
    'run_type': 'benchmark_sample',
    'granularity': 'commune-day',
    'data_source': DAILY_FEATURE_SOURCE,
    'sampling_method': 'all_positives_deterministic_negatives',
})
mlflow.log_params({
    'negative_sample_rate': NEGATIVE_SAMPLE_RATE,
    'random_seed': RANDOM_SEED,
    'train_end': SPLIT_TRAIN_END.date().isoformat(),
    'validation_start': SPLIT_VAL_START.date().isoformat(),
    'test_start': SPLIT_TEST_START.date().isoformat(),
    'n_features': len(X),
    'n_train_sample': len(y_train),
    'n_validation_sample': len(y_val),
    'n_test_sample': len(y_test),
})
mlflow.log_dict({'features': X, 'metadata': dataset_metadata}, 'benchmark_context.json')

search_results = []
for model_name, parameter_grid in model_grids.items():
    for parameters in ParameterGrid(parameter_grid):
        if model_name == 'LightGBM':
            estimator = lgb.LGBMClassifier(
                **parameters,
                scale_pos_weight=search_pos_weight,
                random_state=42,
                n_jobs=-1,
                verbosity=-1,
            )
        else:
            estimator = xgb.XGBClassifier(
                **parameters,
                scale_pos_weight=search_pos_weight,
                tree_method='hist',
                random_state=42,
                n_jobs=-1,
                eval_metric='logloss',
            )

        with mlflow.start_run(run_name=f'grid_{model_name}', nested=True):
            estimator.fit(X_search_train, y_search_train)
            val_predictions = estimator.predict_proba(X_search_val)[:, 1]
            metrics = {
                'pr_auc_validation': average_precision_score(y_search_val, val_predictions),
                'roc_auc_validation': roc_auc_score(y_search_val, val_predictions),
            }
            mlflow.log_params(parameters)
            mlflow.log_params({
                'model_name': model_name,
                'dataset_version': dataset_metadata.get('dataset_version', 'v2'),
                'dataset_sha256': dataset_metadata.get('sha256', ''),
                'grid_train_end': SPLIT_TRAIN_END,
                'validation_year': SPLIT_VAL_YEAR,
            })
            mlflow.log_metrics(metrics)

        search_results.append({
            'model': model_name,
            **parameters,
            **metrics,
        })

results_grid = pd.DataFrame(search_results).sort_values('pr_auc_validation', ascending=False).reset_index(drop=True)
mlflow.log_table(results_grid, 'grid_results.json')
parent_run.end()
results_grid

,model,learning_rate,max_depth,min_child_samples,n_estimators,num_leaves,pr_auc_validation,roc_auc_validation,min_child_weight
0,LightGBM,0.05,-1,50.0,300,31.0,0.154958,0.821142,NaN
1,LightGBM,0.05,8,50.0,300,31.0,0.154276,0.823155,NaN
2,XGBoost,0.05,6,NaN,300,NaN,0.151999,0.807937,1.0
3,XGBoost,0.05,6,NaN,300,NaN,0.151601,0.809137,5.0
4,LightGBM,0.05,-1,50.0,300,63.0,0.149720,0.818484,NaN
5,LightGBM,0.05,8,50.0,300,63.0,0.149564,0.820683,NaN
6,XGBoost,0.05,8,NaN,300,NaN,0.145114,0.801065,5.0
7,XGBoost,0.05,8,NaN,300,NaN,0.143502,0.799223,1.0


In [ ]:
# Réentraînement final sur tout le Train et évaluation Test

best_row = results_grid.iloc[0]
best_model_name = best_row['model']
model_parameters = {}
for parameter_name in model_grids[best_model_name]:
    value = best_row.get(parameter_name)
    if pd.notna(value):
        expected_type = type(model_grids[best_model_name][parameter_name][0])
        model_parameters[parameter_name] = expected_type(value)

# Poids calculé sur l'ensemble Train complet (jusqu'à 2022 inclus)
final_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

if best_model_name == 'LightGBM':
    best_model = lgb.LGBMClassifier(
        **model_parameters,
        scale_pos_weight=final_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )
    mlflow_model_logger = mlflow.lightgbm.log_model
else:
    best_model = xgb.XGBClassifier(
        **model_parameters,
        scale_pos_weight=final_pos_weight,
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    )
    mlflow_model_logger = mlflow.xgboost.log_model

with mlflow.start_run(run_name=f'best_{best_model_name}'):
    # Entraînement sur tout le train disponible (<= SPLIT_TRAIN_END)
    best_model.fit(X_train, y_train)

    val_predictions = best_model.predict_proba(X_val)[:, 1]
    test_predictions = best_model.predict_proba(X_test)[:, 1]

    final_metrics = {
        f'pr_auc_{SPLIT_VAL_YEAR}': average_precision_score(y_val, val_predictions),
        f'roc_auc_{SPLIT_VAL_YEAR}': roc_auc_score(y_val, val_predictions),
        'pr_auc_test': average_precision_score(y_test, test_predictions),
        'roc_auc_test': roc_auc_score(y_test, test_predictions),
    }

    mlflow.log_params({
        **model_parameters,
        'model_name': best_model_name,
        'dataset_version': dataset_metadata.get('dataset_version', 'v2'),
        'dataset_sha256': dataset_metadata.get('sha256', ''),
        'train_end': SPLIT_TRAIN_END,
        'validation_year': SPLIT_VAL_YEAR,
        'test_start': SPLIT_TEST_START,
    })
    mlflow.log_metrics(final_metrics)
    mlflow_model_logger(best_model, name='model')

print(f"Meilleur modèle retenu : {best_model_name}")
print("Métriques finales :", final_metrics)

2026/09/15 11:46:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Meilleur modèle retenu : LightGBM
Métriques finales : {'pr_auc_2023': 0.15303073502914658, 'roc_auc_2023': 0.8136162706145742, 'pr_auc_test': 0.12424775493176246, 'roc_auc_test': 0.7719287570488254}


In [ ]:
print("Tracking URI utilisé :", mlflow.get_tracking_uri())

Tracking URI utilisé : sqlite:////home/coule/Documents/projets/incendies/notebooks/mlflow.db
